# Fully Automated AutoML Pipeline for Network Intrusion Detection

## Objective
The objective of this project is to develop a fully automated machine learning pipeline for anomaly detection using H2O AutoML.

This pipeline includes:

- Automatic preprocessing
- Automatic class balancing using SMOTE
- Automatic feature selection
- Automatic model selection using H2O AutoML
- Cross-dataset evaluation using UNSW-NB15 and NSL-KDD datasets

## Dataset Structure

Primary dataset:
- UNSW-NB15 (training and testing sets)

Secondary dataset:
- NSL-KDD (used for cross-dataset evaluation)

## Tools Used

- Python
- Pandas and NumPy
- Scikit-learn
- Imbalanced-learn (SMOTE)
- H2O AutoML
- Jupyter Notebook

## Pipeline Overview

The pipeline follows these steps:

1. Environment setup
2. Load dataset
3. Data preprocessing
4. Data balancing using SMOTE
5. Automatic feature selection
6. Model training using H2O AutoML
7. Model evaluation
8. Cross-dataset validation

Each step is documented with explanation and implementation.


---
# Step 1 — Environment Setup

## What we are doing
We install/import the libraries required for the automated pipeline and start the H2O runtime.

## Why we are doing it
H2O AutoML runs inside a local H2O cluster (runtime).  
This must be started before we can train any AutoML models.  
We also import tools for data handling (Pandas/NumPy) and data splitting (Scikit-learn).

## Tools used in this step
- **Pandas / NumPy**: data handling and inspection
- **Scikit-learn**: splitting the dataset into train/validation/test
- **H2O + H2OAutoML**: automated model training and selection

## Output of this step
- Successful connection to an H2O cluster
- Confirmed imports with no errors


In [2]:
# Install packages (run once per environment; safe to re-run)
!pip -q install h2o pandas numpy scikit-learn imbalanced-learn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import h2o
from h2o.automl import H2OAutoML

# Start H2O runtime
h2o.init()

print("Environment setup complete.")


Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,48 secs
H2O_cluster_timezone:,Europe/London
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,2 months and 19 days
H2O_cluster_name:,H2O_from_python_sohib_0caiiv
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.589 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


Environment setup complete.


---
# Step 2 — Load the UNSW-NB15 Dataset

## What we are doing
We load the UNSW-NB15 **training** and **testing** CSV files from the project folder.

## Why we are doing it
The dataset is provided as two separate files:
- a training set used to learn patterns (normal vs attack)
- a testing set used to evaluate performance on unseen data

Loading both files early ensures:
- consistent preprocessing later
- a clean pipeline that avoids data leakage

## Tools used
- **Pandas**: reading CSV files and inspecting data

## Output of this step
- `train_df` and `test_df` loaded successfully
- dataset shape, column names, and first few rows displayed


In [3]:
import os

# Because my notebook is inside AutoML/notebooks,
# I will build paths relative to the AutoML folder.
TRAIN_PATH = os.path.join("..", "data", "raw", "UNSW_NB15_training-set.csv")
TEST_PATH  = os.path.join("..", "data", "raw", "UNSW_NB15_testing-set.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Loaded files successfully")
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns (first 15):", list(train_df.columns[:15]))
display(train_df.head())


Loaded files successfully
Train shape: (175341, 45)
Test shape : (82332, 45)

Train columns (first 15): ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss']


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


---
# Step 2.1 — Identify the Target Column (Label)

## What we are doing
We identify which column represents the class label (normal vs attack).

## Why we are doing it
H2O AutoML needs a clear target column `y` to train supervised models.
For UNSW-NB15, common label columns include:
- `label` (binary)
- `attack_cat` (multi-class)

For anomaly detection in a dissertation, the most common setup is:
- binary classification using `label` (0 = normal, 1 = attack)

## Output of this step
- Confirm which target column exists in the dataset


In [4]:
possible_targets = ["label", "attack_cat", "Label", "class"]
found = [c for c in possible_targets if c in train_df.columns]

print("Possible target columns found:", found)

# Quick view of class distribution if 'label' exists
if "label" in train_df.columns:
    print("\nClass distribution in TRAIN (label):")
    print(train_df["label"].value_counts(dropna=False))


Possible target columns found: ['label', 'attack_cat']

Class distribution in TRAIN (label):
label
1    119341
0     56000
Name: count, dtype: int64


---
# Step 2.2 — Target Variable Selection

## What we are doing
We select the target variable for model training.

The UNSW-NB15 dataset provides two possible target columns:

- `label` → binary classification (0 = normal, 1 = attack)
- `attack_cat` → multi-class classification (attack categories)

## Why we are choosing `label`
This project focuses on anomaly detection, which is typically formulated as a binary classification problem:

- Normal traffic → class 0
- Malicious traffic → class 1

Using the binary `label` column allows the AutoML system to learn to distinguish between normal and anomalous network traffic.

This is the standard approach used in most intrusion detection research.

## Class distribution observed

Training dataset:

- Attack samples: 119,341
- Normal samples: 56,000

This shows class imbalance, which will be addressed later using SMOTE applied only to the training data.

## Output of this step

- Target variable defined as `label`
- Feature matrix and target vector separated


In [6]:
# Define target column
TARGET = "label"

# Separate features and target
X_train_full = train_df.drop(columns=[TARGET])
y_train_full = train_df[TARGET]

X_test_full = test_df.drop(columns=[TARGET])
y_test_full = test_df[TARGET]

print("Target variable selected:", TARGET)

print("\nTraining feature shape:", X_train_full.shape)
print("Training target shape :", y_train_full.shape)

print("\nTesting feature shape :", X_test_full.shape)
print("Testing target shape  :", y_test_full.shape)


Target variable selected: label

Training feature shape: (175341, 44)
Training target shape : (175341,)

Testing feature shape : (82332, 44)
Testing target shape  : (82332,)


---
# Step 3 — Automatic Feature Type Detection

## What we are doing
We automatically detect which features are:

- Numerical features (continuous values such as bytes, packets, duration)
- Categorical features (discrete values such as protocol, service, state)

## Why we are doing it
Machine learning models treat numerical and categorical features differently.

For example:

- Numerical feature → dur = 0.12 seconds
- Categorical feature → proto = TCP, UDP

H2O AutoML requires categorical features to be marked as factors.

Instead of manually selecting these features, we use automatic detection based on the dataset data types.

This ensures:

- Zero manual feature selection
- Fully automated pipeline
- Reproducible and scalable system

## Tools used
- Pandas automatic dtype detection

## Output of this step
- List of numerical features
- List of categorical features
- Feature type summary


In [7]:
# Automatically detect categorical and numerical features

categorical_features = X_train_full.select_dtypes(include=['object']).columns.tolist()
numerical_features = X_train_full.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Automatic feature detection complete\n")

print("Number of total features:", len(X_train_full.columns))
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

print("\nFirst 10 numerical features:")
print(numerical_features[:10])


Automatic feature detection complete

Number of total features: 44
Number of numerical features: 40
Number of categorical features: 4

Categorical features:
['proto', 'service', 'state', 'attack_cat']

First 10 numerical features:
['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload']


---
# Step 4 — Automatic Removal of Non-Feature Columns

## What we are doing
We automatically remove columns that should not be used as input features.

These include:

- Target column (already separated)
- Secondary label columns (such as attack_cat)
- Identifier columns (such as id)

## Why we are doing it

### 1. Removing secondary label column (attack_cat)

The `attack_cat` column contains attack category information and is strongly correlated with the target.

Including it would cause data leakage, allowing the model to cheat and produce unrealistic performance.

### 2. Removing ID column

The `id` column is a unique identifier and contains no useful pattern for prediction.

Machine learning models cannot learn meaningful patterns from identifiers.

## Important note

This removal is performed automatically using rule-based detection, not manual feature selection.

This ensures:

- No human bias
- Fully automated preprocessing pipeline

## Output of this step

- Clean feature set ready for automated modelling


In [8]:
# Automatically detect columns to remove

columns_to_remove = []

# Remove ID column automatically
if "id" in X_train_full.columns:
    columns_to_remove.append("id")

# Remove secondary label column automatically
if "attack_cat" in X_train_full.columns:
    columns_to_remove.append("attack_cat")

print("Columns identified for removal:", columns_to_remove)

# Remove from training and testing features
X_train = X_train_full.drop(columns=columns_to_remove)
X_test = X_test_full.drop(columns=columns_to_remove)

print("\nColumns removed successfully")

print("\nNew training feature shape:", X_train.shape)
print("New testing feature shape :", X_test.shape)


Columns identified for removal: ['id', 'attack_cat']

Columns removed successfully

New training feature shape: (175341, 42)
New testing feature shape : (82332, 42)


---
# Step 5 — Automatic Conversion to H2O Format

## What we are doing
We convert the cleaned training and testing datasets into H2OFrames, which are required for H2O AutoML.

We also automatically convert categorical features into H2O "factor" type.

## Why we are doing it

H2O AutoML requires data in H2OFrame format because:

- It enables distributed computation
- It allows automatic model training and optimization
- It correctly handles categorical features when marked as factors

If categorical features are not converted properly, the model may treat them as numerical values, leading to incorrect learning.

This conversion ensures correct handling of:

- Protocol types (proto)
- Service types (service)
- Connection states (state)

## Tools used

- H2OFrame (H2O data structure)
- Automatic categorical conversion

## Output of this step

- H2O training frame
- H2O testing frame
- Correct feature types for AutoML training


In [9]:
# Combine features and target for H2O
train_clean = X_train.copy()
train_clean[TARGET] = y_train_full

test_clean = X_test.copy()
test_clean[TARGET] = y_test_full

# Convert to H2OFrame
train_h2o = h2o.H2OFrame(train_clean)
test_h2o = h2o.H2OFrame(test_clean)

# Automatically detect categorical columns again (after removal)
categorical_features = train_clean.select_dtypes(include=['object']).columns.tolist()

print("Categorical features detected:", categorical_features)

# Convert categorical columns to factor
for col in categorical_features:
    train_h2o[col] = train_h2o[col].asfactor()
    test_h2o[col] = test_h2o[col].asfactor()

# Convert target to factor (classification)
train_h2o[TARGET] = train_h2o[TARGET].asfactor()
test_h2o[TARGET] = test_h2o[TARGET].asfactor()

print("\nConversion to H2OFrame complete")

print("\nH2O training shape:", train_h2o.shape)
print("H2O testing shape :", test_h2o.shape)


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Categorical features detected: ['proto', 'service', 'state']

Conversion to H2OFrame complete

H2O training shape: (175341, 43)
H2O testing shape : (82332, 43)


---
# Step 6 — Automatic Class Balancing (SMOTENC)

## What we are doing
We automatically balance the training dataset using **SMOTENC** (SMOTE for mixed numeric + categorical data).

## Why we are doing it
Standard SMOTE only supports numeric features.  
UNSW-NB15 contains categorical features such as:

- proto (e.g., tcp/udp)
- service (e.g., http/dns)
- state (e.g., SF)

SMOTENC allows oversampling while respecting categorical columns.

## Important: No data leakage
Balancing is applied ONLY to the training data.  
Testing data remains unchanged for fair evaluation.

## Tools used
- imbalanced-learn **SMOTENC**
- automatic detection of categorical feature indices

## Output of this step
- Balanced training dataset (train only)
- Ready for automated feature selection and H2O AutoML


# Step 6.0 — Random Seed Configuration

## What we are doing
We define a fixed random seed value.

## Why we are doing it

Machine learning operations such as:

- SMOTE balancing
- AutoML model training
- Data sampling

use randomness internally.

Setting a fixed seed ensures:

- Reproducibility of results
- Consistent dissertation experiments
- Scientific validity of the pipeline

This allows experiments to be repeated with identical results.


In [14]:
SEED = 42
print("SEED defined:", SEED)



SEED defined: 42


**Note:** SMOTENC is computationally heavier than standard SMOTE because it supports mixed numeric and categorical features. The UNSW-NB15 dataset contains categorical attributes (proto, service, state), therefore SMOTENC was used to avoid invalid numeric conversion and ensure correct oversampling.


In [15]:
from imblearn.over_sampling import SMOTENC

# Work in pandas for SMOTENC
train_pd = train_h2o.as_data_frame()

X_train_sm = train_pd.drop(columns=[TARGET]).copy()
y_train_sm = train_pd[TARGET].copy()

print("Before SMOTENC class distribution:")
print(y_train_sm.value_counts())

# 1) Auto-detect categorical columns (they are still strings in pandas)
cat_cols = X_train_sm.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_train_sm.columns if c not in cat_cols]

print("\nCategorical columns detected for SMOTENC:", cat_cols)

# 2) Convert categorical columns to integer codes automatically (required by SMOTENC)
cat_mappings = {}
for c in cat_cols:
    X_train_sm[c] = X_train_sm[c].astype("category")
    cat_mappings[c] = list(X_train_sm[c].cat.categories)   # store mapping for restore later
    X_train_sm[c] = X_train_sm[c].cat.codes.astype(int)    # missing becomes -1

# 3) Ensure numeric columns are numeric
for c in num_cols:
    X_train_sm[c] = pd.to_numeric(X_train_sm[c], errors="coerce").fillna(0)

# 4) Build categorical indices automatically
cat_indices = [X_train_sm.columns.get_loc(c) for c in cat_cols]

# 5) Apply SMOTENC
smote_nc = SMOTENC(categorical_features=cat_indices, random_state=SEED)
X_balanced, y_balanced = smote_nc.fit_resample(X_train_sm, y_train_sm)

print("\nAfter SMOTENC class distribution:")
print(pd.Series(y_balanced).value_counts())

# 6) Convert back to DataFrame
X_balanced = pd.DataFrame(X_balanced, columns=X_train_sm.columns)

# 7) Restore categorical columns from codes back to strings (so H2O can treat them as factors)
for c in cat_cols:
    categories = cat_mappings[c]
    X_balanced[c] = X_balanced[c].astype(int).apply(
        lambda k: categories[k] if 0 <= k < len(categories) else "UNK"
    )

# 8) Rebuild balanced training dataframe
train_balanced_pd = X_balanced.copy()
train_balanced_pd[TARGET] = y_balanced

# 9) Convert to H2OFrame
train_balanced_h2o = h2o.H2OFrame(train_balanced_pd)

# Convert categorical columns + target to factor for H2O
for c in cat_cols:
    train_balanced_h2o[c] = train_balanced_h2o[c].asfactor()

train_balanced_h2o[TARGET] = train_balanced_h2o[TARGET].asfactor()

print("\nSMOTENC balancing complete")
print("Balanced training shape:", train_balanced_h2o.shape)


C:\Users\sohib\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


Before SMOTENC class distribution:
label
1    119341
0     56000
Name: count, dtype: int64

Categorical columns detected for SMOTENC: ['proto', 'service', 'state']

After SMOTENC class distribution:
label
0    119341
1    119341
Name: count, dtype: int64
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%

SMOTENC balancing complete
Balanced training shape: (238682, 43)


---
# Step 7 — Automatic Feature Selection Using Model-Based Importance

## What we are doing

We automatically select the most important features using a machine learning model.

A Gradient Boosting Machine (GBM) is trained on the balanced training dataset, and feature importance scores are extracted.

The system then automatically selects features based on their importance.

## Why we are doing it

Using all features can:

- increase model complexity
- increase training time
- introduce noise
- reduce generalization performance

Automatic feature selection improves:

- model efficiency
- training speed
- predictive performance

## Important: Fully automated process

No features are manually selected.

Feature importance is determined automatically by the model.

## Tools used

- H2O Gradient Boosting Machine (GBM)
- Automatic feature importance extraction

## Output of this step

- Ranked feature importance list
- Automatically selected features


In [16]:
from h2o.estimators import H2OGradientBoostingEstimator

# Define feature columns automatically
feature_columns = [col for col in train_balanced_h2o.columns if col != TARGET]

print("Number of input features:", len(feature_columns))

# Train GBM for automatic importance detection
gbm_selector = H2OGradientBoostingEstimator(
    ntrees=100,
    max_depth=5,
    seed=SEED
)

gbm_selector.train(
    x=feature_columns,
    y=TARGET,
    training_frame=train_balanced_h2o
)

print("\nGBM training complete")

# Get feature importance
importance_df = gbm_selector.varimp(use_pandas=True)

print("\nTop 10 most important features:")
display(importance_df.head(10))

# Automatically select features with importance > 0
selected_features = importance_df[importance_df['relative_importance'] > 0]['variable'].tolist()

print("\nNumber of selected features:", len(selected_features))


Number of input features: 42
gbm Model Build progress: |██████████████████████████████████████████████████████| (done) 100%

GBM training complete

Top 10 most important features:


,variable,relative_importance,scaled_importance,percentage
0,sttl,194777.265625,1.000000,0.695776
1,proto,17782.675781,0.091297,0.063523
2,service,13287.849609,0.068221,0.047466
3,smean,13067.014648,0.067087,0.046678
4,ct_srv_dst,9231.794922,0.047397,0.032977
5,dttl,6342.537109,0.032563,0.022657
6,dmean,5550.980957,0.028499,0.019829
7,ct_state_ttl,4002.477539,0.020549,0.014298
8,ct_srv_src,3684.768799,0.018918,0.013163
9,synack,3397.746094,0.017444,0.012137



Number of selected features: 38
